<a href="https://colab.research.google.com/github/newgirlsly/makeit/blob/main/face_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python matplotlib tensorflow keras
!pip install deepface
!pip install mtcnn
!pip install retina-face
!pip install gdown

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from deepface import DeepFace
import time
from collections import deque
import warnings
warnings.filterwarnings('ignore')

print("🔧 Loading enhanced gender detection models...")

🔧 Loading enhanced gender detection models...


In [ ]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
import json

def setup_camera():
    """Setup camera untuk real-time streaming"""
    js = Javascript('''
    async function setupCamera() {
        const video = document.createElement('video');
        video.style.display = 'block';
        video.width = 640;
        video.height = 480;
        document.body.appendChild(video);

        const stream = await navigator.mediaDevices.getUserMedia({
            video: {
                width: { ideal: 640 },
                height: { ideal: 480 },
                facingMode: 'user'
            }
        });

        video.srcObject = stream;
        await video.play();

        return video;
    }
    ''')
    display(js)
    return eval_js('setupCamera()')

In [ ]:
class EnhancedGenderDetector:
    def __init__(self):
        self.gender_history = deque(maxlen=10)  # Menyimpan history 10 frame terakhir
        self.age_history = deque(maxlen=10)
        self.confidence_threshold = 0.75

    def analyze_frame(self, frame):
        """Analisis frame dengan multiple verification untuk gender"""
        try:
            # Simpan frame sementara
            cv2.imwrite('/tmp/temp_frame.jpg', frame)

            # Analisis dengan multiple backends
            backends = ['mtcnn', 'opencv', 'retinaface']
            gender_predictions = []
            age_predictions = []
            emotion_predictions = []

            for backend in backends:
                try:
                    analysis = DeepFace.analyze(
                        img_path='/tmp/temp_frame.jpg',
                        actions=['age', 'gender', 'emotion'],
                        detector_backend=backend,
                        enforce_detection=False,
                        silent=True
                    )

                    if isinstance(analysis, list):
                        analysis = analysis[0]

                    gender_predictions.append(analysis['gender'])
                    age_predictions.append(analysis['age'])
                    emotion_predictions.append(analysis['dominant_emotion'])

                except Exception as e:
                    continue

            if not gender_predictions:
                return None, None, None, 0

            # ENSEMBLE VOTING SYSTEM
            final_gender, final_age, final_emotion, confidence = self._ensemble_voting(
                gender_predictions, age_predictions, emotion_predictions
            )

            # Update history
            self.gender_history.append((final_gender, confidence))
            self.age_history.append(final_age)

            # Apply temporal smoothing
            smoothed_gender, smoothed_confidence = self._temporal_smoothing()
            smoothed_age = int(np.mean(self.age_history))

            return smoothed_gender, smoothed_age, final_emotion, smoothed_confidence

        except Exception as e:
            print(f"Analysis error: {e}")
            return None, None, None, 0

    def _ensemble_voting(self, gender_preds, age_preds, emotion_preds):
        """Sistem voting ensemble dengan weighted confidence"""
        from collections import Counter

        # Gender voting dengan confidence weighting
        man_total = 0
        woman_total = 0
        total_weight = 0

        for gender in gender_preds:
            man_conf = gender['Man']
            woman_conf = gender['Woman']
            weight = max(man_conf, woman_conf) / 100.0  # Normalize weight

            if man_conf > woman_conf:
                man_total += weight
            else:
                woman_total += weight
            total_weight += weight

        if man_total > woman_total:
            final_gender = "Pria"
            confidence = (man_total / total_weight) * 100
        else:
            final_gender = "Wanita"
            confidence = (woman_total / total_weight) * 100

        # Age averaging
        final_age = int(np.mean(age_preds))

        # Emotion voting
        emotion_counter = Counter(emotion_preds)
        final_emotion = emotion_counter.most_common(1)[0][0]

        return final_gender, final_age, final_emotion, confidence

    def _temporal_smoothing(self):
        """Smoothing temporal untuk hasil yang lebih stabil"""
        if not self.gender_history:
            return "Unknown", 0

        # Hitung confidence-weighted average
        total_confidence = 0
        gender_scores = {"Pria": 0, "Wanita": 0}

        for gender, confidence in self.gender_history:
            gender_scores[gender] += confidence
            total_confidence += confidence

        if total_confidence == 0:
            return "Unknown", 0

        # Normalize scores
        for gender in gender_scores:
            gender_scores[gender] /= total_confidence

        # Pilih gender dengan score tertinggi
        final_gender = max(gender_scores.items(), key=lambda x: x[1])[0]
        final_confidence = gender_scores[final_gender] * 100

        return final_gender, final_confidence

    def get_confidence_level(self, confidence):
        """Tentukan level confidence"""
        if confidence > 80:
            return "Tinggi", (0, 255, 0)  # Hijau
        elif confidence > 65:
            return "Sedang", (0, 255, 255)  # Kuning
        else:
            return "Rendah", (0, 0, 255)  # Merah

In [ ]:
class RealTimeFaceAnalyzer:
    def __init__(self):
        self.detector = EnhancedGenderDetector()
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    def draw_enhanced_info(self, frame, faces, gender, age, emotion, confidence):
        """Gambar informasi enhanced pada frame"""
        for (x, y, w, h) in faces:
            # Determine color based on confidence
            confidence_level, color = self.detector.get_confidence_level(confidence)

            # Draw face bounding box
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)

            # Draw info background
            info_height = 120
            cv2.rectangle(frame, (x, y - info_height), (x + w, y), color, -1)
            cv2.rectangle(frame, (x, y - info_height), (x + w, y), color, 2)

            # Prepare info text
            info_lines = [
                f"Gender: {gender}",
                f"Usia: {age} tahun",
                f"Ekspresi: {emotion}",
                f"Confidence: {confidence:.1f}%",
                f"Level: {confidence_level}"
            ]

            # Draw info text
            for i, text in enumerate(info_lines):
                y_text = y - info_height + 25 + (i * 20)
                cv2.putText(frame, text, (x + 5, y_text),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            # Draw confidence bar
            bar_width = w
            bar_height = 10
            confidence_width = int((confidence / 100) * bar_width)
            cv2.rectangle(frame, (x, y + h + 5), (x + bar_width, y + h + 5 + bar_height), (100, 100, 100), -1)
            cv2.rectangle(frame, (x, y + h + 5), (x + confidence_width, y + h + 5 + bar_height), color, -1)

        return frame

    def run_real_time_analysis(self):
        """Jalankan real-time analysis"""
        print("🚀 Starting Real-Time Face Analysis...")
        print("📋 Instructions:")
        print("   - Pastikan wajah terlihat jelas")
        print("   - Pencahayaan yang baik")
        print("   - Wajah menghadap kamera")
        print("   - Tekan 'q' untuk keluar")
        print("\n")

        # Setup camera
        video = setup_camera()
        time.sleep(2)  # Biarkan kamera initialize

        frame_count = 0
        start_time = time.time()

        while True:
            try:
                # Capture frame dari JavaScript
                js = Javascript('''
                async function captureFrame(video) {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    const ctx = canvas.getContext('2d');
                    ctx.drawImage(video, 0, 0);
                    return canvas.toDataURL('image/jpeg', 0.8);
                }
                ''')
                display(js)
                frame_data = eval_js('captureFrame(video)')

                # Decode frame
                frame_bytes = b64decode(frame_data.split(',')[1])
                frame_array = np.frombuffer(frame_bytes, dtype=np.uint8)
                frame = cv2.imdecode(frame_array, cv2.IMREAD_COLOR)

                if frame is None:
                    continue

                # Convert BGR to RGB untuk display
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Deteksi wajah untuk bounding box
                gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
                faces = self.face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))

                # Analisis frame
                gender, age, emotion, confidence = self.detector.analyze_frame(frame_rgb)

                # Update FPS counter
                frame_count += 1
                elapsed_time = time.time() - start_time
                fps = frame_count / elapsed_time if elapsed_time > 0 else 0

                # Draw informasi
                if gender and len(faces) > 0:
                    frame_rgb = self.draw_enhanced_info(frame_rgb, faces, gender, age, emotion, confidence)

                # Draw FPS
                cv2.putText(frame_rgb, f"FPS: {fps:.1f}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

                # Draw status
                status = f"Wajah terdeteksi: {len(faces)}" if len(faces) > 0 else "Tidak ada wajah"
                cv2.putText(frame_rgb, status, (10, 60),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

                # Display frame
                plt.figure(figsize=(12, 8))
                plt.imshow(frame_rgb)
                plt.axis('off')
                plt.title('REAL-TIME FACE ANALYSIS - Enhanced Gender Detection',
                         fontsize=14, fontweight='bold', pad=10)
                plt.tight_layout()
                plt.show()

                # Clear output untuk frame berikutnya
                display.clear_output(wait=True)

                # Check for stop condition
                try:
                    from IPython import get_ipython
                    # User bisa stop dengan interrupt kernel
                    pass
                except:
                    break

            except KeyboardInterrupt:
                print("\n🛑 Real-time analysis dihentikan oleh user")
                break
            except Exception as e:
                print(f"❌ Error: {e}")
                continue

        print(f"📊 Statistik: {frame_count} frame diproses dalam {elapsed_time:.1f} detik")
        print(f"📈 Average FPS: {fps:.1f}")

In [ ]:
class GenderFeatureAnalyzer:
    """Analisis fitur wajah untuk koreksi gender tambahan"""

    def __init__(self):
        self.jawline_threshold = 0.7
        self.eyebrow_threshold = 0.6

    def analyze_facial_features(self, face_roi):
        """Analisis fitur wajah untuk membantu koreksi gender"""
        try:
            gray = cv2.cvtColor(face_roi, cv2.COLOR_RGB2GRAY)
            height, width = gray.shape

            # Analisis rasio wajah (jawline)
            jaw_width = width
            face_height = height
            jaw_ratio = jaw_width / face_height

            # Analisis area mata (pria cenderung memiliki alis lebih tebal)
            eye_region = gray[int(height*0.2):int(height*0.4), int(width*0.2):int(width*0.8)]
            eyebrow_intensity = np.mean(eye_region)

            # Rules berdasarkan fitur
            masculine_features = 0
            if jaw_ratio > self.jawline_threshold:
                masculine_features += 1
            if eyebrow_intensity < 100:  # Alis lebih gelap/tebal
                masculine_features += 1

            return masculine_features >= 1  # True jika cenderung masculine

        except Exception as e:
            return None

In [ ]:
# Main execution
def main():
    print("🎯 ENHANCED REAL-TIME FACE ANALYSIS")
    print("=" * 50)
    print("\nFitur Utama:")
    print("✅ Real-time gender detection dengan ensemble voting")
    print("✅ Temporal smoothing untuk hasil stabil")
    print("✅ Confidence-based gender correction")
    print("✅ Multiple model verification")
    print("✅ Real-time FPS monitoring")
    print("\n")

    # Initialize analyzer
    analyzer = RealTimeFaceAnalyzer()

    # Jalankan real-time analysis
    try:
        analyzer.run_real_time_analysis()
    except Exception as e:
        print(f"❌ Error utama: {e}")
        print("💡 Tips: Pastikan kamera diizinkan dan refresh halaman jika ada masalah")

# Jalankan program
if __name__ == "__main__":
    main()

🎯 ENHANCED REAL-TIME FACE ANALYSIS

Fitur Utama:
✅ Real-time gender detection dengan ensemble voting
✅ Temporal smoothing untuk hasil stabil
✅ Confidence-based gender correction
✅ Multiple model verification
✅ Real-time FPS monitoring


🚀 Starting Real-Time Face Analysis...
📋 Instructions:
   - Pastikan wajah terlihat jelas
   - Pencahayaan yang baik
   - Wajah menghadap kamera
   - Tekan 'q' untuk keluar




<IPython.core.display.Javascript object>

❌ Error utama: NotAllowedError: Permission denied
💡 Tips: Pastikan kamera diizinkan dan refresh halaman jika ada masalah


In [ ]:
!pip install opencv-python matplotlib tensorflow keras
!pip install deepface

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from deepface import DeepFace
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import json

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
def analyze_face_with_age_emotion():
  # Mengambil foto dari kamera
  filename = take_photo()

  # Baca gambar dengan OpenCV
  img = cv2.imread(filename)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  # Analisis wajah menggunakan DeepFace
  try:
    analysis = DeepFace.analyze(img_path=filename,
                               actions=['age', 'gender', 'emotion', 'race'],
                               enforce_detection=True)

    # Jika multiple faces terdeteksi, ambil yang pertama
    if isinstance(analysis, list):
        analysis = analysis[0]

    # Dapatkan koordinat wajah
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    # Gambar bounding box dan informasi
    for (x, y, w, h) in faces:
      # Gambar rectangle di sekitar wajah
      cv2.rectangle(img_rgb, (x, y), (x+w, y+h), (0, 255, 0), 2)

      # Tambahkan teks informasi
      info_text = [
          f"Usia: {analysis['age']} tahun",
          f"Gender: {analysis['dominant_gender']}",
          f"Ekspresi: {analysis['dominant_emotion']}",
          f"Ras: {analysis['dominant_race']}"
      ]

      # Tampilkan informasi di atas bounding box
      for i, text in enumerate(info_text):
        y_text = y - 10 - (i * 25)
        cv2.putText(img_rgb, text, (x, y_text if y_text > 20 else y + h + 20 + (i * 25)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    # Tampilkan hasil analisis detail
    print("=== HASIL ANALISIS WAJAH ===")
    print(f"Perkiraan Usia: {analysis['age']} tahun")
    print(f"Gender: {analysis['dominant_gender']} (Akurasi: {analysis['gender'][analysis['dominant_gender']]:.2f})")
    print(f"Ekspresi Dominan: {analysis['dominant_emotion']}")
    print("\nDetail Ekspresi:")
    for emotion, score in analysis['emotion'].items():
      print(f"  {emotion}: {score:.2f}%")

    print("\nDetail Ras:")
    for race, score in analysis['race'].items():
      print(f"  {race}: {score:.2f}%")

    # Tampilkan gambar
    plt.figure(figsize=(12, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title('Hasil Analisis Wajah - Deteksi Umur dan Ekspresi')
    plt.show()

  except Exception as e:
    print(f"Error dalam analisis wajah: {e}")
    print("Pastikan wajah terlihat jelas dalam foto")

In [ ]:
def analyze_multiple_faces():
  # Mengambil foto dari kamera
  filename = take_photo()

  # Baca gambar dengan OpenCV
  img = cv2.imread(filename)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  try:
    # Analisis semua wajah dalam gambar
    analyses = DeepFace.analyze(img_path=filename,
                               actions=['age', 'gender', 'emotion'],
                               enforce_detection=True,
                               detector_backend='opencv')

    # Deteksi wajah untuk mendapatkan koordinat
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    print(f"Terdeteksi {len(analyses)} wajah")

    # Untuk setiap wajah yang terdeteksi
    for i, (analysis, (x, y, w, h)) in enumerate(zip(analyses, faces)):
      # Gambar rectangle dengan warna berbeda untuk setiap wajah
      color = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)][i % 4]
      cv2.rectangle(img_rgb, (x, y), (x+w, y+h), color, 3)

      # Tambahkan informasi untuk setiap wajah
      info_text = [
          f"Wajah {i+1}",
          f"Usia: {analysis['age']}tahun",
          f"Gender: {analysis['dominant_gender']}",
          f"Ekspresi: {analysis['dominant_emotion']}"
      ]

      # Tampilkan informasi
      for j, text in enumerate(info_text):
        y_text = y - 10 - (j * 25)
        cv2.putText(img_rgb, text, (x, y_text if y_text > 20 else y + h + 20 + (j * 25)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

      # Print detail untuk setiap wajah
      print(f"\n=== WAJAH {i+1} ===")
      print(f"Usia: {analysis['age']} tahun")
      print(f"Gender: {analysis['dominant_gender']}")
      print(f"Ekspresi: {analysis['dominant_emotion']}")

    # Tampilkan gambar dengan semua wajah
    plt.figure(figsize=(12, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Deteksi {len(analyses)} Wajah - Analisis Umur dan Ekspresi')
    plt.show()

  except Exception as e:
    print(f"Error: {e}")

In [42]:
# Untuk analisis satu wajah dengan detail lengkap
print("=== ANALISIS SATU WAJAH ===")
analyze_face_with_age_emotion()

# Untuk analisis multiple wajah
# print("=== ANALISIS MULTIPLE WAJAH ===")
# analyze_multiple_faces()

=== ANALISIS SATU WAJAH ===


<IPython.core.display.Javascript object>

Error dalam analisis wajah: Face could not be detected in photo.jpg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Pastikan wajah terlihat jelas dalam foto
